In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
 
 
def tj(vm, vs, v0_max, v2_max):
    val = math.sqrt(v2_max * (vm - vs))
    if val <= v0_max:
        return val / v2_max
    else:
        return v0_max / v2_max
 
 
def ta(vm, vs, v0_max, v2_max):
    val = math.sqrt(v2_max * (vm - vs))
    tj_val = tj(vm, vs, v0_max, v2_max)
    if val <= v0_max:
        return 2 * tj_val
    else:
        return (vm - vs) / v0_max + tj_val
 
 
def t_minus_j(vm, ve, v0_max, v2_max):
    val = math.sqrt(v2_max * (vm - ve))
    if val <= v0_max:
        return val / v2_max
    else:
        return v0_max / v2_max
 
 
def td(vm, ve, v0_max, v2_max):
    val = math.sqrt(v2_max * (vm - ve))
    t_minus_j_val = t_minus_j(vm, ve, v0_max, v2_max)
    if val <= v0_max:
        return 2 * t_minus_j_val
    else:
        return (vm - ve) / v0_max + t_minus_j_val
 
 
def travel_distance(t_a, t_d, T, vm, vs=0):
    """
    Placeholder for integral of distance traveled over time intervals.
    Replace this function with your actual integral calculation:
    integral_0_to_ta + integral_T_minus_td_to_T.
 
    For demonstration, we use a dummy function assuming constant velocity:
    distance = vm * (t_a + t_d)
    Adjust or replace as needed.
    """
    s1=1/2*(vm+vs)*t_a  # distance during acceleration
    s2 = 1 / 2 * (vm + vs) * t_d  # distance during constant velocity
 
    return s1 + s2
 
 
def optimize_vm(v_max, vs, ve, v0_max, v2_max, l, delta_l, T):
    vm = v_max
    delta_v = v_max / 2
 
    while True:
        t_j_val = tj(vm, vs, v0_max, v2_max)
        t_a_val = ta(vm, vs, v0_max, v2_max)
        t_j_minus_val = t_minus_j(vm, ve, v0_max, v2_max)
        t_d_val = td(vm, ve, v0_max, v2_max)
 
        dist = travel_distance(t_a_val, t_d_val, T, vm)
 
        # Flowchart condition 1
        if dist <= l:
            # Flowchart condition 2
            if vm == v_max:
                break
            else:
                if (l - delta_l) <= dist:
                    break
                else:
                    vm += delta_v
        else:
            vm -= delta_v
 
        delta_v /= 2
 
        # Optional: stop if delta_v too small to avoid infinite loop
        if delta_v < 1e-6:
            break
 
    return vm, t_a_val, t_d_val, t_j_val, t_j_minus_val
 
 
# Example parameters - replace these with your actual values
v_maxx = 2.4  # Maximum velocity
vs = 0.0  # Start velocity
ve = 0.0  # End velocity
v0_max = 4.2  # Some max velocity constant from equations
v2_max = 20000  # Another velocity-related constant
l = 0.15  # Total allowed distance or travel length
delta_l = 0.0001  # Threshold for termination judgment
T = 3.0  # Total time (used in travel_distance)
 
# Run optimization
vm_opt, ta_opt, td_opt, tj_opt, t_minus_j_opt = optimize_vm(
    v_maxx, vs, ve, v0_max, v2_max, l, delta_l, T
)
 
print("Optimized S-curve velocity profile:")
print(f"vm = {vm_opt:.6f}")
print(f"ta = {ta_opt:.6f}, td = {td_opt:.6f}")
print(f"tj = {tj_opt:.6f}, t-j = {t_minus_j_opt:.6f}")
 
# time in the linear motion phase
t_const = T - ta_opt - td_opt
print(f"Time in constant velocity phase: t_const = {t_const:.6f}")
 
# plotting the S-curve profile
 
time_points = np.linspace(0, T, 1000)
velocity_profile = []
t_array = []
for t in time_points:
    vss = vs + (v0_max * ((ta_opt - tj_opt) - tj_opt)) + (1 / 2) * v0_max**2 / v2_max
    vss1 = (
        vm_opt
        - ((v0_max**2) / (2 * v2_max))
        - v0_max * ((T - t_minus_j_opt) - (T - td_opt + t_minus_j_opt))
    )
    if t <= tj_opt:
        v = vs + (1 / 2) * v2_max * (t**2)
        velocity_profile.append(v)
        t_array.append(t)
 
    elif tj_opt <= t <= (ta_opt - tj_opt):
        v = vs + (v0_max * (t - tj_opt)) + (1 / 2) * v0_max**2 / v2_max
        velocity_profile.append(v)
        t_array.append(t)
 
    elif (ta_opt - tj_opt) <= t <= ta_opt:
        v = vss - (1 / 2) * v2_max * (t - (ta_opt - tj_opt)) ** 2 + v0_max * (t - (ta_opt - tj_opt))
        velocity_profile.append(v)
        t_array.append(t)
 
    elif ta_opt <= t <= (T - td_opt):
        v = vm_opt
        velocity_profile.append(v)
        t_array.append(t)
 
    elif (T - td_opt) <= t <= (T - td_opt + t_minus_j_opt):
        v = vm_opt- (1 / 2) * v2_max * (t**2 - (T - td_opt) ** 2) + v2_max * (T - td_opt) * (t - (T - td_opt))
        velocity_profile.append(v)
        t_array.append(t)
 
    elif (T - td_opt + t_minus_j_opt) <= t <= (T - t_minus_j_opt):
        v = (
            vm_opt
            - ((v0_max**2) / (2 * v2_max))
            - v0_max * (t - (T - td_opt + t_minus_j_opt))
        )
        velocity_profile.append(v)
        t_array.append(t)
    elif (T - t_minus_j_opt) <= t <= T:
        v = vss1 + (
            (1 / 2) * v2_max * (t**2 - (T - t_minus_j_opt) ** 2)
            - v2_max * (T - t_minus_j_opt) * (t - (T - t_minus_j_opt))
        ) - v0_max * (t - (T - t_minus_j_opt))
        velocity_profile.append(v)
        t_array.append(t)
 
#remove data points where velocity is greater than vm_opt
plt.scatter(t_array, velocity_profile, label="Velocity Profile")
plt.title("Optimized S-Curve Velocity Profile")
plt.xlabel("Time (s)")
plt.ylabel("Velocity (mm/s)")
plt.grid()
plt.show()